# 00_setup — môi trường Kaggle cho mamba-vi-keyboard
Owner: TN. Chạy lại toàn bộ notebook ở đầu mỗi phiên.

**Trước khi chạy** (Settings / Add Input):
1. Accelerator: **GPU T4** · Internet: **On** (tài khoản đã xác minh số điện thoại).
2. Add Input → Datasets: `mamba-btl-data` (DL) và `mamba-btl-wheels` (MB).
3. Muốn resume: Add Input output (Save Version) của lần chạy trước.

Dữ liệu đọc trực tiếp từ `/kaggle/input/mamba-btl-data`, không cần chép. Checkpoint và log ghi vào `/kaggle/working/{checkpoints,logs}` (ngoài thư mục clone).

In [ ]:
# 1. Kiểm tra GPU và phiên bản
!nvidia-smi
!python -c "import torch,sys; print(torch.__version__, torch.version.cuda, sys.version)"

In [ ]:
# 2. Xoá bản clone cũ, clone repo (đổi BRANCH để thử nhánh của PR)
BRANCH = "main"
%cd /kaggle/working
!rm -rf mamba-vi-keyboard
!git clone -q -b {BRANCH} https://github.com/daisyngocha/mamba-vi-keyboard.git
%cd /kaggle/working/mamba-vi-keyboard
!git log --oneline -1

In [ ]:
# 3. Cài mamba-ssm + causal-conv1d từ wheel đã lưu (MB), không biên dịch lại
import glob, pathlib
wheels = sorted(glob.glob("/kaggle/input/mamba-btl-wheels/**/*.whl", recursive=True))
print("\n".join(wheels) or "Chưa có wheel — xem quyết định backend Mamba trong docs/decisions.md")
notes = pathlib.Path("/kaggle/input/mamba-btl-wheels/install_notes.txt")
if notes.exists():
    print(notes.read_text())
if wheels:
    !pip install -q --no-deps {" ".join(wheels)}
!pip install -q pyyaml pytest

In [ ]:
# 4. Smoke test: mamba-ssm (nếu có) + pipeline + kiểm tra giao diện
try:
    import torch
    from mamba_ssm import Mamba
    m = Mamba(d_model=64, d_state=16, d_conv=4, expand=2).cuda()
    x = torch.randn(2, 128, 64, device="cuda", requires_grad=True)
    m(x).sum().backward()
    print("mamba-ssm OK")
except Exception as e:
    print("mamba-ssm chưa dùng được:", repr(e))
!bash scripts/smoke_test.sh
!python -m pytest -q tests

In [ ]:
# 5. Kiểm tra dữ liệu
!ls -la /kaggle/input/mamba-btl-data
!cat /kaggle/input/mamba-btl-data/meta.json

## Chạy một lượt huấn luyện
Tên lượt tự đặt `<model>_L<len>_s<seed>`. Lượt dài: **Save Version → Save & Run All** để chạy nền (tối đa 12 giờ) và giữ checkpoint làm output. Ghi lượt chạy vào bảng theo dõi.

In [ ]:
# 6. Ví dụ (bỏ comment khi cần)
# !python train.py --config configs/mamba.yaml --set data.seq_len=256 --set seed=0
# !python train.py --config configs/transformer.yaml --set data.seq_len=256 --set data.train_file=sample.bin --set train.train_chars=5_000_000